# 島根での汎化性能

**学習のノートブックと分けてある。** 理由は見た目ではない ──
同居していたせいで、学習側だけ config に直して
**評価側が直書きのまま残った**（`docs/設計図_現状.md` の `2.3`）。
分ければ両方が `build_loaders` を通る。

- **島根の主系統は `subset: all`**（全24,569枚）。
  ここで作る分割は使わない。実世界に当てたときの模倣だから、
  基準は土俵をそろえる前である（`docs/リサーチマップ.md`）
- `gtonly`（警戒区域ありだけ）は補助。混ぜて語らないこと

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "dc5lib" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

from dc5lib import ckpt
from dc5lib.evaluate import evaluate
from dc5lib.pipeline import build_loaders, seed_everything
from dc5lib.registry import get
from dc5lib.traincfg import train_cfg

CONDITION = "__CONDITION__"
REGION = "Shimane"

cond = get(CONDITION)
CFG = train_cfg(CONDITION)
seed_everything(CFG.seed)
print(cond.display())
print()
print("--- 使う重みの素性 ---")
print(ckpt.describe(cond.weights))

In [ ]:
# ============================================================
# 島根の pkl を読む。**学習と同じ手順**（build_loaders）を通る。
# 分割も作られるが、主系統では使わない（下のセルで全域を測る）。
# ============================================================
ld = build_loaders(cond, REGION)

In [ ]:
# ============================================================
# **全域で測る。** 島根は subset: all が主系統なので、
# 学習・検証・テストの3つを合わせた全タイルで評価する。
# ============================================================
from torch.utils.data import ConcatDataset, DataLoader

allset = ConcatDataset([ld.datasets[k] for k in ("train", "val", "test")])
ld.test = DataLoader(allset, batch_size=CFG.batch_size, shuffle=False)
print(f"全域 {len(allset):,} 枚で測る"
      f"（警戒区域あり {ld.n_used:,} / 背景 {ld.n_bg:,} は入らない）")
res_all = evaluate(cond, ld, "test")

In [ ]:
# ============================================================
# 参考：島根の「テスト側」だけ。**広島と同じ切り方にした場合**の値。
# 主系統ではない。混ぜて語らないこと。
# ============================================================
ld2 = build_loaders(cond, REGION, verbose=False)
res_te = evaluate(cond, ld2, "test")

## 次にやること

オーバーラップ推論での箇所評価（`42`〜`44` の数字はこちら）。

```bash
python analysis/run_overlap_eval.py --region both --conditions <条件名> --record
python analysis/run_overlap_prune.py --region shimane --conditions <条件名>
```

和集合に入れるなら、先に分割の一致を確かめる。

```bash
python tools/audit_splits.py --region both --conditions <全構成員>
```